# Lecture 7: Root-finding

You will learn to solve non-linear equations numerically (**scipy.optimize**).

**The plan:**

1. **Newton's method** (sections 2-4): use the slope of $f$ to jump towards the root. Fast, but needs a derivative - analytical or numerical - and a good starting point.
2. **Bisection** (section 5): only needs a sign change on an interval. Safe, but slow.
3. **Scipy** (section 6): the implementations you should actually use, and the settings that matter.
4. **The consumer problem** (sections 7-8): first order conditions *are* non-linear equations. We solve a single one derived by hand, and then the full system in one go.

Along the way: what happens with **multiple roots**, and how to choose a **tolerance**.

**Table of contents**<a id='toc0_'></a>    
- 1. [Introduction](#toc1_)    
- 2. [Derivative based methods](#toc2_)    
- 3. [Multiple roots](#toc3_)    
- 4. [Numerical derivative](#toc4_)    
- 5. [Derivative free methods: Bisection](#toc5_)    
- 6. [Root-Finding using Scipy](#toc6_)    
- 7. [The Consumer Problem (again)](#toc7_)    
  - 7.1. [Newton's Method](#toc7_1_)    
  - 7.2. [Bisection](#toc7_2_)    
- 8. [Both First Order Conditions](#toc8_)    
  - 8.1. [Fewer equations](#toc8_1_)    
- 9. [Takeaways](#toc9_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'-'})
plt.rcParams.update({'font.size': 14})
from scipy import optimize

import ipywidgets as widgets # interactive figures

**scipy.optimize:** [overview](https://docs.scipy.org/doc/scipy/reference/optimize.html) + [tutorial](https://docs.scipy.org/doc/scipy/tutorial/optimize.html)

## 1. <a id='toc1_'></a>[Introduction](#toc0_)

In economics, we really like setting **First Order Conditions** to 0. We do it every time we want to solve for optimal behavior.

Therefore, we often want to **solve non-linear equations** of the form

$$ 
f(x) = 0, \quad x \in \mathbb{R} 
$$

There are 2 types of method:
* **Derivative based:** Require more inputs, but give fast convergence.
* **Derivative free:** Require fewer inputs, but final convergence often  slow.

A simple **example** of a function for our root finding:

$$
f(x) = -x^3 + 2x^2 + 4x + 30, \qquad f^{\prime}(x) = -3x^2 + 4x + 4
$$

**Theoretical result:** $f^{\prime}(x) = 0$ at $x=-\frac{2}{3}$ and $x=2$, where $f(-\frac{2}{3}) \approx 28.5 > 0$ and $f(2) = 38 > 0$. Together with $f(x) \rightarrow +\infty$ for $x \rightarrow -\infty$ and $f(x) \rightarrow -\infty$ for $x \rightarrow +\infty$, this implies that $f$ crosses zero **exactly once**, and does so above $x=2$.

The root is $x^{\ast} = 4.43084740\dots$ (the two remaining roots of the cubic are complex, see `np.roots([-1,2,4,30])`).

## 2. <a id='toc2_'></a>[Derivative based methods](#toc0_)

**Newton's method**: 

We use a **first order** Taylor approximation of the function at a **new point** $x_1$:

$$ 
f(x_1) \approx f(x_0) + f^{\prime}(x_0)(x_1-x_0)
$$

We choose $x_1$ where the Taylor approximation is zero: 

$$
f(x_1) = 0 \Leftrightarrow x_1 = x_0 - \frac{f(x_0)}{f^{\prime}(x_0)}
$$

**Algorithm:** `find_root()`

1. Choose tolerance $\epsilon > 0$, guess on $x_0$ and set $k = 0$.
2. Calculate $f(x_k)$ and $f^\prime(x_k)$
3. If $|f(x_k)| < \epsilon$ then stop.
4. Calculate new candidate $x_{k+1} = x_k - \frac{f(x_k)}{f^{\prime}(x_k)}$.
5. Set $k = k + 1$ and return to step 2.

In [2]:
def find_root(x0,f,df,max_iter=500,tol=1e-8,full_info=False):
    """ find root
        
    Args:
    
        x0 (float): initial value
        f (callable): function
        df (callable): derivative
        max_iter (int): maximum number of iterations
        tol (float): tolerance
        full_info (bool): controls information returned
        
    Returns:
    
        x(s) (float/ndarray): root (if full_info, all x tried)
        k (int): number of iterations used
        fs (ndarray): function values used (if full_info) 
        fps (ndarray): derivative values used (if full_info)
        
    """

    # a. initialize
    xs, fs, fps = [], [], [] # history of x, f(x), f'(x)
    x_k = x0 # initial guess
    k = 0 # iteration counter

    # b. main loop
    while True:
        
        # step 2: evaluate function
        x = x_k
        fx  = f(x)
        dfx = df(x)

        xs.append(x); fs.append(fx); fps.append(dfx)

        # step 3: check convergence
        if abs(fx) < tol:
            break
        elif k >= max_iter:
            raise ValueError(f'Maximum number of iterations {max_iter} reached.')
            
        # step 4: update
        x_k = x - fx/dfx
        
        # step 5: increment
        k += 1
        
    if full_info:
        return np.array(xs), k, np.array(fs), np.array(fps)
    else:
        return x, k

In [3]:
def plot_find_root(x0,f,fp,xmin=-8,xmax=8,xn=100):
    
    # a. find root and return all information 
    x,max_iter,fs,fps = find_root(x0,f,df=fp,full_info=True)
    
    # b. compute function on grid
    xvec = np.linspace(xmin,xmax,xn)
    fxvec = f(xvec)
    
    # c. figure
    def _figure(k):
        
        # i. here we have the linear (first order) taylor approximation (=tangent line at x_k)
        fapprox = fs[k] + fps[k]*(xvec-x[k]) 
            
        # ii. figure
        fig = plt.figure(figsize=(15,5))
        ax = fig.add_subplot(1,1,1)
        
        ax.plot(xvec,fxvec) # the true f(x)
        ax.plot(x[k],0,'o',color='blue',mfc='none',label='$x_{k}$')#  current iterate x_k on x-axis
        ax.plot(xvec,fapprox) # tangent line at x_k
        ax.plot(x[k],fs[k],'o',color='black',label='$f(x_k)$') # the point (x_k, f(x_k))       

        ax.axhline(0,ls='-',lw=1,color='black') # coordinate system
        ax.axvline(0,ls='-',lw=1,color='black') # coordinate system
        
        # if there IS a next iterate, plot it
        if k+1 < len(x):
            ax.axvline(x[k+1],ls='--',lw=1,color='black') # cross zero: dashed vertical line
            ax.plot(x[k+1],0,'o',color='green',mfc='none',label='$x_{k+1}$') # next

        # cosmetics        
        ax.legend(loc='upper left',facecolor='white',ncols=3,frameon=True)
        ax.set_ylim(min([fxvec[-1],fxvec[0]]),max([fxvec[-1],fxvec[0]]))

        fig.tight_layout()
    
    # this gives us an iteration slider to animate the method.
    widgets.interact(_figure,
        k=widgets.IntSlider(description='iterations', min=0, max=max_iter, step=1, value=0)
    );

Run and plot:

In [4]:
f  = lambda x: -x**3 + 2*x**2 + 4*x + 30
df = lambda x: -3*x**2 + 4*x + 4
x0 = -5.0

x, k = find_root(x0, f, df)
print(f'x = {x:.8f} [iterations: {k}]')
print(f'f(x) = {f(x):.2e}')

x = 4.43084740 [iterations: 7]
f(x) = -2.33e-10


In [5]:
plot_find_root(x0,f,df)

interactive(children=(IntSlider(value=0, description='iterations', max=7), Output()), _dom_classes=('widget-in…

## 3. <a id='toc3_'></a>[Multiple roots](#toc0_)

Newton's method finds **a** root - which one depends on where we start.

Example with **three roots**:

$$
\begin{align*}
f(x) &= x^3-3x^2-x+3 = (x+1)(x-1)(x-3) \\
f^{\prime}(x) &= 3x^2-6x-1
\end{align*}
$$

In [6]:
f_multi  = lambda x: x**3 - 3*x**2 - x + 3
df_multi = lambda x: 3*x**2 - 6*x - 1

for x0 in [-2.0,0.5,2.0,2.5,4.0]:
    x,k = find_root(x0,f_multi,df_multi)
    print(f'x0 = {x0:5.1f} -> x = {x:6.2f} [iterations: {k}]')

x0 =  -2.0 -> x =  -1.00 [iterations: 5]
x0 =   0.5 -> x =   1.00 [iterations: 3]
x0 =   2.0 -> x =  -1.00 [iterations: 1]
x0 =   2.5 -> x =   3.00 [iterations: 5]
x0 =   4.0 -> x =   3.00 [iterations: 5]


In [ ]:
plot_find_root(2.0,f_multi,df_multi,xmin=-3,xmax=5)

**Note:** Newton's method has **no global guarantee**. We normally end at a nearby root, but not always: from $x_0=2$ the tangent is almost flat ($f^{\prime}$ is close to zero at $x \approx 2.15$), so the first step jumps far to the left and we end at $x=-1$ instead of $x=1$ or $x=3$.

**Implication:** The starting value is a *choice*, and in economic models it should be an informed one.

## 4. <a id='toc4_'></a>[Numerical derivative](#toc0_)

Sometimes, you might not have the **analytical derivative**. Then, you can instead use the **numerical derivative**.

**Numerical derivative:** Define $\Delta$ to be a small number, then we approximate the derivative by 
$$
 \frac{df}{dx} \approx \frac{f(x+\Delta) - f(x)}{\Delta}
$$

In [ ]:
# a. numerical derivative (forward)
Delta = 1e-8
fp_approx = lambda x: (f(x+Delta)-f(x))/Delta

# b. find root
x0 = -5.0
x,k = find_root(x0,f,fp_approx,max_iter=200)   
print(f'x = {x:.8f} [iterations: {k}]')
print(f'f(x) = {f(x):.2e}')

## 5. <a id='toc5_'></a>[Derivative free methods: Bisection](#toc0_)

If $f$ is continuous and $f(a)f(b)<0$, then $f$ changes sign on $[a,b]$ and must cross zero somewhere in between (the intermediate value theorem).

**Bisection** uses this: set $m=\tfrac{a+b}{2}$ and keep the half-interval where the sign change persists. Each step halves the interval, so we get a guaranteed and steadily tightening bracket around a root - slower than Newton, but very reliable.

**Caveat:** A sign change guarantees *at least* one root (there may be more, as in section 3), and roots where $f$ only *touches* zero are never found.

**Algorithm:** `bisection()`

1. Set $a_0 = a$ and $b_0 = b$ where $f(a)$ and $f(b)$ have opposite sign, $f(a_0)f(b_0)<0$
2. Compute $f(m_0)$ where $m_0 = (a_0 + b_0)/2$ is the midpoint.
3. Determine the next sub-interval $[a_1,b_1]$:
  * If $f(a_0)f(m_0) < 0$ (different signs) then $a_1 = a_0$ and $b_1 = m_0$ (i.e. focus on $[a_0,m_0]$).
  * If $f(m_0)f(b_0) < 0$ (different signs) then $a_1 = m_0$ and $b_1 = b_0$ (i.e. focus on $[m_0,b_0]$).
4. Repeat steps 2–3 until $|f(m_k)| < \epsilon$.

In [ ]:
def bisection(f,a,b,max_iter=500,tol=1e-8,full_info=False):
    """ bisection
    
    Solve equation f(x) = 0 for a <= x <= b.
    
    Args:
    
        f (callable): function
        a (float): left bound
        b (float): right bound
        max_iter (int): maximum number of iterations
        tol (float): tolerance on solution
        full_info (bool): controls information returned
        
    Returns:
    
        m (float/ndarray): root (if full_info, all x tried)
        k (int): number of iterations used
        a (ndarray): left bounds used
        b (ndarray): right bounds used
        fm (ndarray): function values at midpoints
        
    """

    # i. check that f(a) and f(b) have opposite signs
    fa = f(a)
    fb = f(b)
    
    if fa*fb >= 0:
        raise ValueError(f'Function has same sign at endpoints a={a} and b={b}.')

    # ii. initialize
    a_l, b_l, m_l, fm_l = [], [], [], [] # history of a, b, m, f(m)
    k = 0

    # iii. main loop
    while True:
        
        # step 2: midpoint
        m  = (a+b)/2
        fm = f(m)
        
        a_l.append(a); b_l.append(b); m_l.append(m); fm_l.append(fm)
        
        # step 3: keep half with sign change
        if abs(fm) < tol:
            break   
        elif k >= max_iter:
            raise ValueError(f'Maximum number of iterations {max_iter} reached.')     
        elif fa*fm < 0:  # keep left half
            b = m
            fb = fm
        elif fb*fm < 0:  # keep right half
            a = m
            fa = fm
        else: # catch-all: this should never happen
            raise ValueError('bisection method fails.')
        
        k += 1
        
    if full_info:
        return np.array(m_l), k, np.array(a_l), np.array(b_l), np.array(fm_l)
    else:
        return m, k

In [ ]:
m,k = bisection(f,-8,8)
print(f'm = {m:.8f}[iterations: {k} ]')
print(f'f(m) = {f(m):.2e}')

**Same result** as before, but **trade-off** between more iterations and no evaluation of derivatives.

In [ ]:
def plot_bisection(f,a,b,xmin=-8,xmax=8,xn=100):
    
    # a. find root and return all information 
    m,max_iter,a,b,fm = bisection(f,a,b,full_info=True)
    
    # b. compute function on grid
    xvec = np.linspace(xmin,xmax,xn)
    fxvec = f(xvec)
    
    # c. figure
    def _figure(k):
        fig = plt.figure(figsize=(15,5))
        ax = fig.add_subplot(1,1,1)
        
        ax.plot(xvec,fxvec)
        ax.axhline(0, color='black')
        ax.plot(m[k],fm[k],'o',color='black',label='current')
        ax.plot([a[k],b[k]],[fm[k],fm[k]],'--',color='green',label='range')
        ax.axvline(a[k],ls='--',color='green')
        ax.axvline(b[k],ls='--',color='green')        
        
        ax.legend(loc='lower left',facecolor='white',frameon=True)
        ax.set_ylim(min([fxvec[-1],fxvec[0]]),max([fxvec[-1],fxvec[0]]))
    
    widgets.interact(_figure,
        k=widgets.IntSlider(description='iterations', min=0, max=max_iter-1, step=1, value=0)
    );

plot_bisection(f,-8,8)

**Note:** Bisection is not good at the final convergence steps. Generally true for methods not using derivatives.

## 6. <a id='toc6_'></a>[Root-Finding using Scipy](#toc0_)

Scipy, naturally, has better implementations of the above algorithms. 

You will most likely want to go for these when doing your own model solution.

**Newton:**

In [ ]:
result = optimize.root_scalar(f,x0=-5.0,fprime=df,method='newton')
print(result)

**Newton without a derivative:** If `fprime` is not given, scipy uses the **secant method**, which - in the same spirit as the numerical derivative in section 4 - replaces $f^{\prime}(x_k)$ with the slope through the last two iterates:

In [ ]:
result = optimize.root_scalar(f,x0=-5.0,method='newton')
print(result)

**Bisect:**

In [ ]:
result = optimize.root_scalar(f,bracket=[-8,8],method='bisect')
print(result)

The **best choice** is the more advanced **Brentq-method** (combines bisection and interpolation methods; thus robustness and speed):

In [ ]:
result = optimize.root_scalar(f,bracket=[-8,8],method='brentq')
print(result)

**Main settings** in `optimize.root_scalar(f,...)`:

| setting | what it does |
| :-- | :-- |
| `x0` | starting value for the derivative based methods |
| `fprime` | the derivative $f^{\prime}$; if omitted, a secant method is used |
| `bracket=[a,b]` | interval with a sign change; required by the derivative free methods |
| `method` | `'newton'`, `'bisect'`, `'brentq'` |
| `xtol`, `rtol` | absolute and relative tolerance **on $x$**: stop when the step (or the bracket) is smaller than `xtol + rtol*abs(x)` |
| `maxiter` | maximum number of iterations |
| `args` | tuple of extra arguments passed on to `f` (e.g. parameters) |

The result contains `.root`, `.converged`, `.flag`, `.iterations` and `.function_calls`. **Always check `.converged`** - a solver that gave up still returns a number.

**What is a reasonable tolerance?** Two different stopping criteria are in play:

* **On the function value**, $|f(x_k)| < \epsilon$ - what our own `find_root()` and `bisection()` use.
* **On the change in $x$**, $|x_{k+1}-x_k| <$ `xtol` $+$ `rtol` $\cdot |x_k|$ - what scipy uses.

They are linked through the slope at the root,

$$
|x_k-x^{\ast}| \approx \frac{|f(x_k)|}{|f^{\prime}(x^{\ast})|}
$$

Here $f^{\prime}(x^{\ast}) \approx -37$, so $|f| < 10^{-8}$ means $x$ is accurate to about $3 \cdot 10^{-10}$. For a **flat** function it is the other way around: $f$ can be tiny while $x$ is still far off.

**Rules of thumb:**

1. **`1e-8` is a good default.** Floating points carry ~16 digits, and $f$ itself is only accurate to $\sim 10^{-16}$ *relative* to its own scale, so below $10^{-12}$-ish you are just chasing rounding noise.
2. **Scale matters.** $|f| < 10^{-8}$ means something completely different if $f$ is measured in millions rather than in units. Decide what accuracy you need *on $x$*, in units you understand economically, and set the tolerance from there.

In [ ]:
# a. reference value: the real root of the cubic (see section 1)
roots = np.roots([-1,2,4,30])
x_true = roots[np.isreal(roots)].real[0]

# b. vary the tolerance
for xtol in [1e-4,1e-8,1e-12]:
    result = optimize.root_scalar(f,bracket=[-8,8],method='brentq',xtol=xtol)
    print(f'xtol = {xtol:.0e}: x = {result.root:.12f}, error = {abs(result.root-x_true):.2e} [iterations: {result.iterations}]')

**Note:** Each extra iteration buys several digits, so a tight tolerance is *cheap* once we are close - the cost of root-finding is in getting to the neighborhood of the root, not in the last digits. This is why the default is tight. Bisection is the exception (section 5): there every digit costs the same 3-4 iterations.

## 7. <a id='toc7_'></a>[The Consumer Problem (again)](#toc0_)

Same Cobb-Douglas consumer as in the optimization lecture,

$$
\max_{x_1,x_2 \geq 0} x_1^{\alpha}x_2^{1-\alpha} \quad \text{s.t.} \quad p_1x_1+p_2x_2 \leq I,\quad 0<\alpha<1
$$

**Step 1: Substitute the budget constraint.** Utility is increasing in both goods, so the budget binds and $x_2$ is a function of $x_1$,

$$
x_2(x_1) = \frac{I-p_1x_1}{p_2} \qquad \Rightarrow \qquad \frac{dx_2}{dx_1} = -\frac{p_1}{p_2}
$$

**Step 2: Differentiate.** Buying one more unit of good 1 gives $\frac{\partial u}{\partial x_1}$, but costs $\frac{p_1}{p_2}$ units of good 2, each worth $\frac{\partial u}{\partial x_2}$. The first order condition is a **single non-linear equation** in $x_1$,

$$
g(x_1) \equiv \frac{\partial u}{\partial x_1} + \frac{\partial u}{\partial x_2}\frac{dx_2}{dx_1} = \alpha x_1^{\alpha-1}x_2^{1-\alpha} - (1-\alpha)\frac{p_1}{p_2}x_1^{\alpha}x_2^{-\alpha} = 0
$$

for $0 < x_1 < \frac{I}{p_1}$ and with $x_2 = x_2(x_1)$ everywhere.

**Step 3: Differentiate once more** to get the derivative Newton needs. Using $x_2 + \frac{p_1}{p_2}x_1 = \frac{I}{p_2}$ it collapses to

$$
g^{\prime}(x_1) = -\alpha(1-\alpha)\frac{I^2}{p_2^2}x_1^{\alpha-2}x_2^{-\alpha-1} < 0
$$

So $g$ is strictly decreasing, and it runs from $+\infty$ (at $x_1 \rightarrow 0$, where good 1 is infinitely valuable) to $-\infty$ (at $x_1 \rightarrow I/p_1$, where good 2 is) $\Rightarrow$ there is **exactly one root**, and both Newton and bisection will find it.

**Analytical solution** (to check against): $x_1^{\ast} = \frac{\alpha I}{p_1}$ and $x_2^{\ast} = \frac{(1-\alpha) I}{p_2}$.

In [ ]:
# a. parameters
alpha = 0.5  # preference weight on good 1
p1 = 1.0     # price of good 1
p2 = 2.0     # price of good 2
I = 10.0     # income

# b. implied x2 from the budget constraint
x2_of = lambda x1: (I-p1*x1)/p2

# c. first order condition and its derivative
g  = lambda x1: alpha*x1**(alpha-1)*x2_of(x1)**(1-alpha) - (1-alpha)*(p1/p2)*x1**alpha*x2_of(x1)**(-alpha)
dg = lambda x1: -alpha*(1-alpha)*(I/p2)**2*x1**(alpha-2)*x2_of(x1)**(-alpha-1)

# d. analytical solution
x1_ana = alpha*I/p1
x2_ana = (1-alpha)*I/p2
print(f'analytical: x1 = {x1_ana:.8f}, x2 = {x2_ana:.8f}')

### 7.1. <a id='toc7_1_'></a>[Newton's Method](#toc0_)

We can reuse `find_root()` from section 2 directly - it only needs $g$ and $g^{\prime}$.

In [ ]:
# a. solve
x0 = 0.4*(I/p1) # initial guess (must be interior)
x1,k = find_root(x0,g,dg)
x2 = x2_of(x1)

# b. report
print(f'Newton: x1 = {x1:.8f}, x2 = {x2:.8f} [iterations: {k}]')
print(f'error on x1: {abs(x1-x1_ana):.2e}')

### 7.2. <a id='toc7_2_'></a>[Bisection](#toc0_)

Bisection needs a bracket $[a,b]$ with $g(a)g(b)<0$. Since $g$ runs from $+\infty$ to $-\infty$, **any** $a$ just above $0$ and $b$ just below $I/p_1$ works.

In [ ]:
# a. solve
a = 1e-4
b = 9.7 # note: not exactly I/p1, see below
x1,k = bisection(g,a,b)
x2 = x2_of(x1)

# b. report
print(f'Bisection: x1 = {x1:.8f}, x2 = {x2:.8f} [iterations: {k}]')
print(f'error on x1: {abs(x1-x1_ana):.2e}')

**Many more iterations than Newton** - the price of not using the derivative.

**Note:** The bracket $b=9.7$ is deliberately asymmetric. With the symmetric bracket $[\epsilon,I/p_1-\epsilon]$ the very first midpoint is $\approx I/(2p_1) = 5 = x_1^{\ast}$ (because $\alpha = 0.5$), and bisection would stop immediately. Tight, economically motivated bounds do help a lot - but do not count on being that lucky.

## 8. <a id='toc8_'></a>[Both First Order Conditions](#toc0_)

In section 7 we did the elimination **by hand**: the budget constraint removed $x_2$, so we were left with a single equation. But we do not have to. We can also write down **all** the first order conditions and hand the whole system to the solver.

The Lagrangian is

$$
\mathcal{L} = x_1^{\alpha}x_2^{1-\alpha} + \lambda\left(I-p_1x_1-p_2x_2\right)
$$

The two first order conditions plus the budget constraint give

$$
h(x_1,x_2,\lambda) = \begin{bmatrix}
\alpha x_1^{\alpha-1}x_2^{1-\alpha} - \lambda p_1 \\
(1-\alpha) x_1^{\alpha}x_2^{-\alpha} - \lambda p_2 \\
I - p_1 x_1 - p_2 x_2
\end{bmatrix} = \begin{bmatrix} 0 \\ 0 \\ 0 \end{bmatrix}
$$

This is still root-finding, just with $h:\mathbb{R}^3 \rightarrow \mathbb{R}^3$: **3 equations in 3 unknowns**. Newton's method generalizes directly,

$$
z_{k+1} = z_k - J(z_k)^{-1}h(z_k),\qquad z = (x_1,x_2,\lambda)
$$

where $J$ is the Jacobian (the matrix of partial derivatives). `optimize.root` does this for us, and approximates $J$ numerically if we do not supply it.

In [ ]:
def h(z):
    """ first order conditions, z = (x1,x2,lambda) """

    x1,x2,lam = z

    return np.array([
        alpha*x1**(alpha-1)*x2**(1-alpha) - lam*p1, # dL/dx1 = 0
        (1-alpha)*x1**alpha*x2**(-alpha) - lam*p2,  # dL/dx2 = 0
        I - p1*x1 - p2*x2                           # budget constraint
        ])

# a. initial guess (must be interior, h is undefined for x1,x2 <= 0)
z0 = np.array([0.4*I/p1,0.4*I/p2,1.0])

# b. solve
result = optimize.root(h,z0)
print(result.message)

# c. report
x1,x2,lam = result.x
print(f'x1 = {x1:.8f}, x2 = {x2:.8f}, lambda = {lam:.8f} [function calls: {result.nfev}]')
print(f'error on x1: {abs(x1-x1_ana):.2e}')

Finer tolerance:

In [ ]:
# a. solve
result = optimize.root(h,z0,tol=1e-12)
print(result.message)

# c. report
x1,x2,lam = result.x
print(f'x1 = {x1:.8f}, x2 = {x2:.8f}, lambda = {lam:.8f} [function calls: {result.nfev}]')
print(f'error on x1: {abs(x1-x1_ana):.2e}')

**Main settings** in `optimize.root(fun,x0,...)`:

| setting | what it does |
| :-- | :-- |
| `x0` | initial guess - **required**; in more than one dimension there is no bracketing to fall back on |
| `jac` | the Jacobian $J$; if omitted it is approximated by finite differences |
| `method` | `'hybr'` (default, a safeguarded Newton) |
| `tol` | tolerance for termination |
| `options` | method specific, e.g. `{'xtol':1e-12,'maxfev':1000}` |
| `args` | tuple of extra arguments passed on to `fun` |

The result contains `.x` (the solution), `.success`, `.message`, `.nfev` and `.fun` (the **residuals**, i.e. how well each equation is actually satisfied - print them when something looks off).

**Note:** With several equations there is no guarantee like the sign change in bisection, so the starting value matters even more, and `.success` is worth checking every single time. Here the default tolerance leaves residuals of $\sim 10^{-10}$; `options={'xtol':1e-12}` takes them to zero at the cost of two extra function calls.

### 8.1. <a id='toc8_1_'></a>[Fewer equations](#toc0_)

Dividing the two first order conditions by each other eliminates $\lambda$ and gives the familiar MRS = price ratio,

$$
\frac{\alpha x_2}{(1-\alpha)x_1} = \frac{p_1}{p_2}
$$

which together with the budget constraint is a system of **2 equations in 2 unknowns**. Eliminating $x_2$ as well brings us all the way back to the single equation in section 7.

In [ ]:
def h_small(z):
    """ first order conditions with lambda eliminated, z = (x1,x2) """

    x1,x2 = z

    return np.array([
        alpha*x2/((1-alpha)*x1) - p1/p2, # MRS = price ratio
        I - p1*x1 - p2*x2                # budget constraint
        ])

# a. solve
result_small = optimize.root(h_small,np.array([0.4*I/p1,0.4*I/p2]))
x1_small,x2_small = result_small.x

# b. report
print(f'2 equations: x1 = {x1_small:.8f}, x2 = {x2_small:.8f} [function calls: {result_small.nfev}]')

# c. check the multiplier from the 3-equation system
u = x1**alpha*x2**(1-alpha)
print(f'lambda = {lam:.8f}, u/I = {u/I:.8f}')

**Trade-off:** Eliminating by hand gives a smaller and better behaved problem (and here even a closed-form solution to check against), but it takes work and must be redone for every new model. Handing the full system of first order conditions to `optimize.root` requires almost no derivations and scales to models where no elimination is possible.

## 9. <a id='toc9_'></a>[Takeaways](#toc0_)

1. **Root-finding** is how we impose first order conditions numerically: solve $f(x)=0$.
2. **Newton's method** is fast, but needs a derivative (analytical or numerical) and a decent starting point - with multiple roots the starting point decides which one we get.
3. **Bisection** needs no derivative and always works given a bracket with a sign change, but is slow - especially in the final steps.
4. **Use scipy**: `optimize.root_scalar(...,method='brentq')` for a bracketed scalar equation and `optimize.root` for a system of equations.
5. You can **substitute by hand** down to one equation, or let the solver take **all the first order conditions** at once. The first is faster and more robust, the second is much less work.